In [ ]:
import osmnx as ox
import networkx as nx
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point, shape
from shapely import wkt
import altair as alt
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import shutil
import subprocess
import os
import folium
import folium.plugins
import datetime
alt.data_transformers.disable_max_rows()
import warnings
import os, glob, time
from tqdm.auto import tqdm
warnings.filterwarnings("ignore")



os.environ["JAVA_HOME"] = subprocess.run(
    ["/usr/libexec/java_home", "-v", "21"],
    capture_output=True, text=True, check=True
).stdout.strip()


from r5py.util.config import Config
import r5py

In [ ]:
neigh = pd.read_csv("Data/Neigh/neigh.csv")

neigh["geometry"] = neigh["geom"].apply(wkt.loads)

neigh = gpd.GeoDataFrame(
    neigh,
    geometry="geometry",
    crs="EPSG:4326"
)
neigh = neigh[["name", "geometry"]]
neigh["geometry"] = neigh.buffer(0)


neigh_layer = alt.Chart(neigh).mark_geoshape(
    stroke="black",
    fill="lightblue"
).properties(
    width=600,
    height=600
    
).encode(tooltip=['name'])

In [6]:
bus_sol = pd.read_csv('Nodes/N-Bus-Sol.csv')
fgc_sol = pd.read_csv('Nodes/N-FGC-Sol.csv')
metro_sol = pd.read_csv('Nodes/N-Metro-Sol.csv')
tram_sol = pd.read_csv('Nodes/N-Tram-Sol.csv')

all_stops = pd.concat([bus_sol, fgc_sol, metro_sol, tram_sol], ignore_index=True)
all_stops['geometry']= all_stops['geometry'].apply(wkt.loads)
all_stops = gpd.GeoDataFrame(all_stops, geometry='geometry', crs="EPSG:4326")


In [10]:
all_stops[all_stops['stop_id'] == 319]

,id,stop_id,name,linia,stop_type,geometry
124,SB-319,319,Entença - Gran Via,IU Stop,Bus,POINT (2.15235 41.37837)
865,SM-319,319,Sants Estació,IU Stop,Metro,POINT (2.14189 41.38096)


In [8]:
all_stops['stop_id'].value_counts().sort_values(ascending=False)

stop_id
319     2
516     2
213     2
214     2
216     2
       ..
PLRL    1
UNIV    1
XILE    1
LLUC    1
ALFS    1
Name: count, Length: 889, dtype: int64

In [ ]:
# periferic_stops = neigh[neigh['name'].isin(['Torre Baró', 'Vallbona','la Marina del Prat Vermell','Vallvidrera, el Tibidabo i les Planes','Ciutat Meridiana'])]
# periferic_stops = gpd.sjoin(all_stops, periferic_stops, how='inner', predicate='within')
# all_stops = all_stops[~all_stops['id'].isin(periferic_stops['id'])]
len(all_stops)

In [ ]:
(((943 * 943) *24)/2433)/3600

In [ ]:
a = alt.Chart(all_stops).mark_geoshape(size=0, color='red').encode(tooltip=['id', 'name']).properties(width=600, height=600)
neigh_layer + a

In [ ]:
# cache_dir = Config().CACHE_DIR
# for f in cache_dir.glob("*.mapdb*"):
#     f.unlink()

In [ ]:
transport_network = r5py.TransportNetwork(
    "Data/Neigh/cataluna-260811.osm.pbf",
    elevation_model =["Data/Neigh/elevacions-terreny-lidar-Catalunya-2m-2008-2011tif1786456535055.tif"]
    )

In [ ]:
glories = shape({
    'type': 'Point',
    'coordinates':  (2.184459824628127, 41.401546206394364)
})

glories= gpd.GeoDataFrame(
    {'id' : ['Glories - City Gate'],
    'geometry': [glories]},
    crs="EPSG:4326"
)
glories

# City Gate to all stops

In [ ]:
all_stops['id'].value_counts()

In [ ]:
process = 0
failed_iteraries = []
from_city_gate = r5py.DetailedItineraries(
            transport_network,
            origins=glories,
            destinations=all_stops,
            transport_modes=[r5py.TransportMode.CAR],
            snap_to_network=True,
            departure=datetime.datetime(2026, 8, 19, 13, 20),

)
from_city_gate.rename(columns={'from_id': 'origen','to_id':'dest'}, inplace=True)
print(len(from_city_gate))
from_city_gate = from_city_gate[from_city_gate['travel_time'] > pd.Timedelta(minutes=5)]
print(len(from_city_gate))
from_city_gate = from_city_gate[from_city_gate['travel_time'] < pd.Timedelta(minutes=20)]
print(len(from_city_gate))
from_city_gate = from_city_gate[['origen','dest','geometry']]
from_city_gate = from_city_gate.merge(all_stops[['id','name']], left_on='dest', right_on='id', how='left')
from_city_gate['tram'] = 'City Gate' + ' - ' + from_city_gate['name']
from_city_gate.drop(columns=['id','name'], inplace=True)
from_city_gate.to_crs('EPSG:25831', inplace=True)
from_city_gate['length'] = from_city_gate['geometry'].length
from_city_gate['speed'] = 18 /3.6 
from_city_gate['time'] = pd.to_timedelta(from_city_gate['length'] / from_city_gate['speed'],unit='s')
print(len(from_city_gate))
from_city_gate = from_city_gate[from_city_gate['time'] > pd.Timedelta(minutes=5)]
print(len(from_city_gate))
from_city_gate = from_city_gate[from_city_gate['time'] < pd.Timedelta(minutes=20)]
print(len(from_city_gate))
from_city_gate['time'] = from_city_gate['time'].apply(
    lambda x: f"{int(x.total_seconds() // 60):02d}:{int(x.total_seconds() % 60):02d}"
)
from_city_gate.to_crs('EPSG:4326', inplace=True)
from_city_gate

In [ ]:
all_stops

In [ ]:
checkpoint_dir = "Data/trajectory_cache"
os.makedirs(checkpoint_dir, exist_ok=True)

stop_ids = all_stops['id'].unique()
chunk_size = 20  # origins per chunk against all destinations — tune based on timing below

for i in tqdm(range(0, len(stop_ids), chunk_size)):
    checkpoint_path = f"{checkpoint_dir}/chunk_{i:06d}.csv"
    if os.path.exists(checkpoint_path):
        print(f"Checkpoint {checkpoint_path} exists, skipping chunk {i}...")
        continue  # already done — safe to resume after interruption

    for stop_type in all_stops['stop_type'].unique():
        stops_for_type = all_stops[all_stops['stop_type'] == stop_type]

        origins = stops_for_type[stops_for_type['id'].isin(stop_ids[i:i + chunk_size])]
        chunk = r5py.DetailedItineraries(
            transport_network,
            origins=origins,
            destinations=all_stops,
            transport_modes=[r5py.TransportMode.CAR],
            snap_to_network=True,
            departure=datetime.datetime(2026, 8, 19, 13, 20),
        )
        chunk.to_csv(checkpoint_path,index =False)

In [ ]:
across_stops = pd.concat(
    [pd.read_csv(f) for f in sorted(glob.glob(f"{checkpoint_dir}/chunk_*.csv"))],
    ignore_index=True
)
across_stops

In [ ]:
across_stops_mod = across_stops.copy()
across_stops_mod['distance'] = across_stops_mod['distance'].astype(float)
across_stops_mod['travel_time'] = pd.to_timedelta(across_stops_mod['travel_time'])
print(len(across_stops_mod))
across_stops_mod = across_stops_mod[across_stops_mod['distance'] > 0]
print(len(across_stops_mod))
across_stops_mod = across_stops_mod[across_stops_mod['travel_time'] > pd.Timedelta(minutes=5)]
print(len(across_stops_mod))
across_stops_mod = across_stops_mod[across_stops_mod['travel_time'] < pd.Timedelta(minutes=20)]
print(len(across_stops_mod))

chunk it

In [ ]:
final_df = pd.DataFrame()

for stop_id in tqdm(across_stops_mod['from_id'].unique()):
    subset = across_stops_mod[across_stops_mod['from_id'] == stop_id]
    subset['geometry'] = subset['geometry'].apply(wkt.loads)
    final_df = pd.concat([final_df, subset], ignore_index=True)


In [ ]:
across_stops_mod = gpd.GeoDataFrame(final_df, geometry='geometry', crs="EPSG:4326")

In [ ]:
alt.data_transformers.disable_max_rows()
itin_layer = alt.Chart(across_stops_mod[:10].drop(columns=['travel_time'])).mark_geoshape(strokeWidth=4, filled = False).properties(width=600, height=600).encode(color = 'to_id:N', tooltip=['from_id', 'to_id', 'distance'])
neigh_layer + itin_layer

In [ ]:
across_stops_mod = across_stops_mod[['from_id', 'to_id', 'geometry']]
across_stops_mod.rename(columns={'from_id': 'origen', 'to_id': 'dest'}, inplace=True)
across_stops_mod.to_crs('EPSG:25831', inplace=True)
across_stops_mod['length'] = across_stops_mod['geometry'].length
across_stops_mod['speed'] = 18 / 3.6 
across_stops_mod['time'] = pd.to_timedelta(across_stops_mod['length'] / across_stops_mod['speed'],unit='s')
print(len(across_stops_mod))
across_stops_mod = across_stops_mod[across_stops_mod['time'] > pd.Timedelta(minutes=5)]
print(len(across_stops_mod))
across_stops_mod = across_stops_mod[across_stops_mod['time'] < pd.Timedelta(minutes=20)]
print(len(across_stops_mod))
across_stops_mod['time'] = across_stops_mod['time'].apply(
    lambda x: f"{int(x.total_seconds() // 60):02d}:{int(x.total_seconds() % 60):02d}"
)
across_stops_mod.to_crs('EPSG:4326', inplace=True)
across_stops_mod = across_stops_mod.merge(all_stops[['id','name']], left_on='origen', right_on='id', how='left')
across_stops_mod.rename(columns={'name': 'origen_name'}, inplace=True)
across_stops_mod = across_stops_mod.merge(all_stops[['id','name']], left_on='dest', right_on='id', how='left')
across_stops_mod.rename(columns={'name': 'dest_name'}, inplace=True)
across_stops_mod['tram'] = across_stops_mod['origen_name'] + ' - ' + across_stops_mod['dest_name']

In [ ]:
#across_stops_mod.drop(columns=['id_x','id_y','origen_name','dest_name'], inplace=True)
across_stops_mod['type'] = 'IU Trajectory'
across_stops_mod = across_stops_mod[['origen', 'dest', 'tram', 'type','length', 'speed', 'time', 'geometry']]

In [ ]:
from_city_gate['type'] = 'IU Trajectory'
from_city_gate = from_city_gate[['origen', 'dest', 'tram', 'type','length', 'speed', 'time', 'geometry']]

In [ ]:
all_routes = pd.concat([from_city_gate, across_stops_mod], ignore_index=True)
all_routes.insert(7, 'directed', True)
all_routes


In [ ]:
all_routes.to_csv('Edges/IU-Bus.csv', index=False)